# Monitored leave-one-stimulus-out RSA: dogs and humans
Runs steps 1 and 11.2 from the participant ZIPs already in Google Drive. Google will request sign-in when Drive is mounted below. One Colab VM handles both cohorts. Each participant has isolated scratch/logs; group comparisons wait for their cohort's step-1 jobs. The supervisor starts with one worker and adds another only after sustained CPU, RAM, GPU, VRAM and disk headroom. Completed omission ZIPs are resumable.


In [ ]:
# @title Connect your Google Drive (complete Google's sign-in prompt)
from google.colab import drive
drive.mount('/content/drive')
%pip -q install nibabel scipy numpy psutil
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Connect a GPU runtime before running this analysis.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# @title Uploaded folders and bounded resource-aware scheduling
DRIVE_ROOT = '/content/drive/MyDrive/rsa_colab'
COHORT_FOLDERS = {'D': DRIVE_ROOT + '/omission_D', 'H': DRIVE_ROOT + '/omission_H'}
OUT_DIR = DRIVE_ROOT + '/omission_analysis'
CODE_OVERRIDE_DIR = DRIVE_ROOT + '/omission_code'  # current small support files; data ZIPs stay unchanged
WORK_DIR = '/content/omission_supervisor'
MAXIMUM_WORKERS = 2
STEP1_BATCH = 128
VOXEL_BATCH = 256
ALLOWED_MISSING_RUNS = ['H:20:1']  # 39 humans for run 1; all 40 for every other run


In [ ]:
# @title Load current support code, or bundled code from newly built packages
from pathlib import Path
import importlib, json, shutil, sys, zipfile
FILES = ('gpu_rsa.py', 'gpu_stimulus_omission.py', 'run_colab_stimulus_omission.py', 'stimulus_contribution.py', 'supervise_stimulus_omission.py', 'launch_stimulus_omission.py')
code = Path('/content/omission_bootstrap')
code.mkdir(parents=True, exist_ok=True)
override = Path(CODE_OVERRIDE_DIR)
if all((override / name).is_file() for name in FILES):
    for name in FILES:
        shutil.copyfile(override / name, code / name)
else:
    package = next(Path(COHORT_FOLDERS['D']).glob('pkg_*.zip'))
    with zipfile.ZipFile(package) as archive:
        for name in FILES:
            if 'code/' + name not in archive.namelist():
                raise ValueError('Upload the current omission_code support folder; existing data ZIPs need no rebuilding.')
            (code / name).write_bytes(archive.read('code/' + name))
sys.path.insert(0, str(code))
sys.modules.pop('launch_stimulus_omission', None)
importlib.invalidate_caches()
import launch_stimulus_omission as launcher


In [ ]:
# @title Launch both cohorts; reruns reconnect to an active supervisor
receipt = launcher.launch(drive_root=DRIVE_ROOT, output_root=OUT_DIR, work=WORK_DIR,
    maximum_workers=MAXIMUM_WORKERS, step1_batch=STEP1_BATCH, voxel_batch=VOXEL_BATCH,
    code_override=str(code), allowed_missing_runs=ALLOWED_MISSING_RUNS)
print('Live logs and status:', OUT_DIR)


In [ ]:
# @title Monitor completion and resource use (interrupting this cell does not kill workers)
import time
status_path = Path(receipt['status'])
while True:
    if status_path.is_file():
        state = json.loads(status_path.read_text())
        resources = state.get('resources', {})
        print('State:', state['state'], 'Active:', list(state.get('active', {})),
              'Completed jobs:', len(state.get('finished', [])),
              'CPU %:', resources.get('cpu_percent'), 'GPU %:', resources.get('gpu_utilization'),
              'RAM free GB:', round(resources.get('ram_available', 0)/2**30, 1), flush=True)
        if state['state'] == 'failed':
            raise RuntimeError(state.get('error', state.get('failed')))
        if state['state'] == 'complete':
            break
    else:
        print('Preflighting cohort packages and masks...', flush=True)
    time.sleep(30)


## Results and resuming
`OUT_DIR/results_omission_D` and `OUT_DIR/results_omission_H` contain the pipeline-relative result ZIPs. Restore them using `tools/unpack_results.py`. The group maps retain `by_stim/run-01/b_GreyMatter2mmB-D-r-4_kendall_DogA1.nii.gz` naming. No step 11.3 is needed.

`OUT_DIR/resources.jsonl`, `status.json`, `supervisor.log` and `logs/` record resources, scheduling, PIDs, progress and failures. The scheduler never exceeds `MAXIMUM_WORKERS`; it reduces concurrency and retries once with smaller batches after CUDA memory pressure. To lower the live cap, edit `OUT_DIR/supervisor_config.json` to `{"maximum_workers": 1}`. Existing workers finish their current jobs.

Reconnect and run the notebook again to resume checkpoints. Only one supervisor is launched per runtime/output folder. Release the Colab runtime after both cohorts finish to avoid idle compute charges.


## Additional step 11.2 classes
After step 1 is complete for the full cohort, reuse class_all checkpoints for class_hum and class_dog. Put run_omission_class_groups.py and the exact class_hum.csv/class_dog.csv files in rsa_colab/omission_code. Human 20 is excluded only for run 1.

In [ ]:
import sys
from pathlib import Path
root = Path('/content/drive/MyDrive/rsa_colab')
sys.path.insert(0, str(root / 'omission_code'))
from run_omission_class_groups import run_class
for specie in ('D', 'H'):
    for rsa_class in ('hum', 'dog'):
        run_class(root / f'omission_{specie}',
                  root / f'omission_analysis/results_omission_{specie}',
                  root / f'omission_analysis_class_tests/results_{specie}',
                  specie, rsa_class, root / f'omission_code/class_{rsa_class}.csv',
                  f'/content/class_groups/{specie}_{rsa_class}')
